# 数理変換タクティクス AI トレーニング
## ハイブリッドEmbedding版（数学特徴 + 学習特徴）

### モデルの構造
```
カード整数
  │
  ├─ 数学Embedding（固定・非学習）
  │    素因数, 約数数, 桁の和, 素数フラグ → 9次元
  │
  └─ 学習Embedding（学習・更新される）
       ゲーム戦略を自動発見 → 16次元
  │
  結合 → 25次元/枚
  │
  GlobalMaxPool（手札全体を25次元に圧縮）
  │
  ×2（自分・相手）＋スカラー5次元 = 55次元
  │
  Dense → 7クラス（操作選択）
```

### 誤差逆伝播の流れ
```
予測結果 → 損失計算 → 誤差を逆向きに伝播

Dense層        ← 誤差あり、重み更新
GlobalMaxPool  ← 誤差あり（勾配通過）
学習Embedding  ← 誤差あり、重み更新  ★
数学Embedding  ← 誤差通過するが重み更新しない（固定）
```
数学的に正しい値は最初から与えるので、
学習Embeddingは「どの数字がゲーム上強いか」だけに集中できる。

In [ ]:
!pip install tensorflowjs scikit-learn -q
print('完了！')

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflowjs as tfjs
import random
from sklearn.model_selection import train_test_split

MAX_HAND    = 20
EMBED_DIM   = 16   # 学習Embeddingの次元数
NUM_ACTIONS = 7
MAX_CARD    = 150
PRIMES      = [2, 3, 5, 7, 11, 13]  # 使用する素数リスト
MATH_DIM    = len(PRIMES) + 3        # 6(素因数) + 1(素数フラグ) + 1(約数数) + 1(桁和) = 9

print(f'TensorFlow: {tf.__version__}')
print(f'数学特徴の次元数: {MATH_DIM}')
print(f'カード1枚あたりの特徴: {MATH_DIM}(数学) + {EMBED_DIM}(学習) = {MATH_DIM+EMBED_DIM}次元')

## 数学特徴テーブルの構築

カード1〜150を9次元の数学的ベクトルに変換する対応表を作ります。

| 特徴 | 次元 | 意味 |
|---|---|---|
| 素因数の指数 | 6次元 | 2で何回割れる？3で？... |
| 大きな素因数フラグ | 1次元 | 13より大きい素因数を持つか |
| 約数の個数 | 1次元 | 何枚の敵カードを攻撃できるか（の指標） |
| 桁の和 | 1次元 | 桁和分裂操作に直接関係 |

In [ ]:
def card_math_features(n):
    """カード1枚を9次元の数学ベクトルに変換"""
    if n == 0:  # パディング
        return [0.0] * MATH_DIM

    orig = n

    # ① 素因数の指数（各素数で何回割り切れるか）
    #    例: 12 = 2²×3¹ → [2,1,0,0,0,0] → 正規化 → [0.5, 0.25, 0, 0, 0, 0]
    factors = []
    for p in PRIMES:
        count = 0
        while n % p == 0:
            n //= p
            count += 1
        factors.append(min(count, 4) / 4.0)  # 最大4乗まで、0〜1に正規化

    # ② 大きな素因数フラグ（13より大きい素数が残っていれば1）
    #    例: 17→1, 7→0（7はPRIMESに含まれるので), 51=3×17→1
    has_large_prime = 1.0 if n > 1 else 0.0

    # ③ 約数の個数（攻撃できる相手カードの多さの指標）
    #    例: 12の約数は1,2,3,4,6,12 → 6個 → 6/20=0.3
    num_divs = sum(1 for d in range(1, orig + 1) if orig % d == 0)
    num_divs_norm = min(num_divs, 20) / 20.0

    # ④ 桁の和（桁和分裂操作に直接使われる値）
    #    例: 123 → 1+2+3=6 → 6/30=0.2
    ds = sum(int(d) for d in str(orig))
    ds_norm = min(ds, 30) / 30.0

    return factors + [has_large_prime, num_divs_norm, ds_norm]

# カード0〜150の数学特徴テーブルを作成（shape: 151 × 9）
FACTOR_TABLE = np.array(
    [card_math_features(i) for i in range(MAX_CARD + 1)],
    dtype=np.float32
)

print(f'数学特徴テーブル shape: {FACTOR_TABLE.shape}')
print()
print('具体例:')
for n in [6, 7, 12, 30, 17]:
    f = card_math_features(n)
    divs = [d for d in range(1, n+1) if n % d == 0]
    ds   = sum(int(d) for d in str(n))
    print(f'  {n:3d}: 素因数={f[:6]} 大素数={f[6]:.0f} 約数数={len(divs)}個({f[7]:.2f}) 桁和={ds}({f[8]:.2f})')

## ゲームシミュレーター

In [ ]:
def gcd(a, b):
    while b: a, b = b, a % b
    return a

def digit_sum(n):
    return sum(int(d) for d in str(n))

class MathCardGame:
    def __init__(self, config=None):
        self.config = config or {
            'initHandCount': 5, 'initMaxNum': 10,
            'drawCount': 2, 'drawMaxNum': 20,
            'handLimitNum': MAX_CARD, 'winScore': 100, 'maxTurns': 10,
        }

    def reset(self):
        cfg = self.config
        self.hands = {
            'P1': [random.randint(1, cfg['initMaxNum']) for _ in range(cfg['initHandCount'])],
            'P2': [random.randint(1, cfg['initMaxNum']) for _ in range(cfg['initHandCount'])],
        }
        self.scores  = {'P1': 0, 'P2': 0}
        self.turn_count    = 1
        self.current_player = 'P1'
        self.done   = False
        self.winner = None

    def opponent(self, pid): return 'P2' if pid == 'P1' else 'P1'

    def get_inputs(self, pid):
        my  = self.hands[pid]
        enm = self.hands[self.opponent(pid)]
        pad = lambda arr: (arr[:MAX_HAND] + [0]*MAX_HAND)[:MAX_HAND]
        scalars = [
            self.scores[pid] / 100.0,
            self.scores[self.opponent(pid)] / 100.0,
            self.turn_count / 10.0,
            len(my)  / MAX_HAND,
            len(enm) / MAX_HAND,
        ]
        return pad(my), pad(enm), scalars

    def best_attack(self, pid):
        enm = self.hands[self.opponent(pid)]
        best, bg = None, 0
        for i, a in enumerate(self.hands[pid]):
            if a == 1: continue
            g = sum(n for n in enm if n % a == 0)
            if g > bg: bg = g; best = (i, a, g)
        return best

    def best_add(self, pid):
        h, lim = self.hands[pid], self.config['handLimitNum']
        best, bv = None, -1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                r = h[i]+h[j]
                if r<=lim and r>bv: bv=r; best=(i,j,r)
        return best

    def best_sub(self, pid):
        h = self.hands[pid]; best, bv = None, 0
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                r = h[i]-h[j]
                if r>0 and r>bv: bv=r; best=(i,j,r)
        return best

    def best_divprod(self, pid):
        h, lim = self.hands[pid], self.config['handLimitNum']
        best, bv = None, -1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j or h[j]==0: continue
                r = (h[i]//h[j])*(h[i]%h[j])
                if r>0 and r<=lim and r>bv: bv=r; best=(i,j,r)
        return best

    def best_dsd(self, pid):
        h, lim = self.hands[pid], self.config['handLimitNum']
        best, bv = None, -1
        for i, num in enumerate(h):
            ds = digit_sum(num)
            if ds==0: continue
            q, r = divmod(num, ds)
            if q<=lim and q>bv: bv=q; best=(i,q,r)
        return best

    def best_gm(self, pid):
        h, lim = self.hands[pid], self.config['handLimitNum']
        best, bv = None, -1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                g = gcd(h[i], h[j])
                if g<=1: continue
                r = h[i]*g
                if r<=lim and r>bv: bv=r; best=(i,j,r)
        return best

    def get_optimal_action(self, pid):
        atk = self.best_attack(pid)
        add = self.best_add(pid)
        sub = self.best_sub(pid)
        dp  = self.best_divprod(pid)
        dsd = self.best_dsd(pid)
        gm  = self.best_gm(pid)
        ms  = self.scores[pid]
        win = self.config['winScore']

        if atk and self.turn_count>1 and ms+atk[2]>=win: return 1
        if gm  and gm[2]>=30:  return 6
        if atk and self.turn_count>1 and atk[2]>=10: return 1
        if add and add[2]>=20: return 2
        if dp  and dp[2]>=15:  return 4
        if atk and self.turn_count>1: return 1
        if gm  and gm[2]>=10: return 6
        if dsd and dsd[1]>=5:  return 5
        if add: return 2
        if dp:  return 4
        if sub: return 3
        return 0

    def execute_action(self, pid, action):
        h   = self.hands[pid]
        eid = self.opponent(pid)
        if action == 1:
            atk = self.best_attack(pid)
            if atk:
                i,v,gain = atk
                self.hands[eid] = [n for n in self.hands[eid] if n%v!=0]
                h.pop(i); self.scores[pid] += gain
                if self.scores[pid] >= self.config['winScore']:
                    self.done = True; self.winner = pid
        elif action == 2:
            a = self.best_add(pid)
            if a: i,j,r=a; h=[c for k,c in enumerate(h) if k!=i and k!=j]; h.append(r)
        elif action == 3:
            s = self.best_sub(pid)
            if s: i,j,r=s; h=[c for k,c in enumerate(h) if k!=i and k!=j]; h.append(r)
        elif action == 4:
            d = self.best_divprod(pid)
            if d: i,j,r=d; h=[c for k,c in enumerate(h) if k!=i and k!=j]; h.append(r)
        elif action == 5:
            d = self.best_dsd(pid)
            if d: idx,q,r=d; h.pop(idx); h.append(q); (h.append(r) if r>0 else None)
        elif action == 6:
            g = self.best_gm(pid)
            if g: i,j,r=g; h=[c for k,c in enumerate(h) if k!=i and k!=j]; h.append(r)
        self.hands[pid] = h
        self.current_player = eid
        if self.current_player == 'P1':
            self.turn_count += 1
            if self.turn_count > self.config['maxTurns']:
                self.done = True
                self.winner = max(self.scores, key=lambda p: self.scores[p])

print('ゲームシミュレーター準備完了！')

## トレーニングデータ生成

In [ ]:
NUM_EPISODES = 30000

my_hands_data, enemy_hands_data, scalars_data, y_data = [], [], [], []
game = MathCardGame()

for ep in range(NUM_EPISODES):
    game.reset()
    while not game.done:
        pid    = game.current_player
        my_h, enm_h, sc = game.get_inputs(pid)
        action = game.get_optimal_action(pid)
        my_hands_data.append(my_h)
        enemy_hands_data.append(enm_h)
        scalars_data.append(sc)
        y_data.append(action)
        game.execute_action(pid, action)
    if (ep+1) % 10000 == 0:
        print(f'{ep+1}/{NUM_EPISODES} 完了 ... データ数: {len(y_data)}')

my_hands_data    = np.array(my_hands_data,    dtype=np.int32)
enemy_hands_data = np.array(enemy_hands_data, dtype=np.int32)
scalars_data     = np.array(scalars_data,     dtype=np.float32)
y_data           = np.array(y_data,           dtype=np.int32)

print(f'\n総データ数: {len(y_data)}')
names = ['パス','攻撃','足し算','引き算','商×余','桁和分裂','GCD掛け']
u, c = np.unique(y_data, return_counts=True)
for i, cnt in zip(u, c):
    print(f'  {i}({names[i]}): {cnt}件 ({cnt/len(y_data)*100:.1f}%)')

## モデル構築（ハイブリッドEmbedding）

### 誤差逆伝播の詳細

```
【順伝播】
カード整数 → 数学Embedding(固定) ─┐
           → 学習Embedding      ─┴─ 結合 → MaxPool → Dense → 予測

【損失計算】
予測: [攻撃:0.3, 足し算:0.6, ...]
正解: [攻撃:1.0, 足し算:0.0, ...]
損失 = -log(0.3) = 1.2  ← 大きいほど間違い

【逆伝播】損失を小さくするための勾配を逆向きに計算
Dense層の重み         ← 「もっと攻撃を選べ」という信号で更新
MaxPool               ← 勾配が通過するだけ（重みなし）
学習Embedding[6]      ← 「6は攻撃に使える」と更新  ★
数学Embedding[6]      ← 勾配は来るが trainable=False なので更新しない
```

In [ ]:
from tensorflow.keras.utils import to_categorical

idx = np.arange(len(y_data))
train_idx, val_idx = train_test_split(idx, test_size=0.1, random_state=42)

def make_dataset(indices, batch_size=256, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((
        {'my_hand': my_hands_data[indices],
         'enemy_hand': enemy_hands_data[indices],
         'scalars': scalars_data[indices]},
        to_categorical(y_data[indices], NUM_ACTIONS)
    ))
    if shuffle: ds = ds.shuffle(10000)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_idx)
val_ds   = make_dataset(val_idx, shuffle=False)

# ── 入力 ──
my_hand_in    = tf.keras.Input(shape=(MAX_HAND,), dtype='int32',   name='my_hand')
enemy_hand_in = tf.keras.Input(shape=(MAX_HAND,), dtype='int32',   name='enemy_hand')
scalar_in     = tf.keras.Input(shape=(5,),        dtype='float32', name='scalars')

# ── 数学Embedding（固定・非学習） ──
# FACTOR_TABLEをそのまま重みとして使う。trainable=Falseで更新されない
math_emb = tf.keras.layers.Embedding(
    input_dim  = MAX_CARD + 1,
    output_dim = MATH_DIM,
    weights    = [FACTOR_TABLE],   # 素因数・約数数・桁和を初期値として設定
    trainable  = False,            # ← 誤差逆伝播で更新しない
    mask_zero  = True,
    name       = 'math_emb'
)

# ── 学習Embedding（更新される） ──
# ゲーム戦略（どの数字がいつ強いか）を自動発見
learned_emb = tf.keras.layers.Embedding(
    input_dim  = MAX_CARD + 1,
    output_dim = EMBED_DIM,
    mask_zero  = True,
    name       = 'learned_emb'
)

def encode_hand(hand_input):
    """手札を数学特徴+学習特徴の結合ベクトルに変換し、MaxPoolで圧縮"""
    math    = math_emb(hand_input)     # (batch, MAX_HAND, 9)  固定
    learned = learned_emb(hand_input)  # (batch, MAX_HAND, 16) 更新
    combined = tf.keras.layers.Concatenate(axis=-1)([math, learned])  # (batch, MAX_HAND, 25)
    return tf.keras.layers.GlobalMaxPooling1D()(combined)             # (batch, 25)

my_pool    = encode_hand(my_hand_in)    # (batch, 25)
enemy_pool = encode_hand(enemy_hand_in) # (batch, 25)

# 25 + 25 + 5 = 55次元
x = tf.keras.layers.Concatenate(name='combined')([my_pool, enemy_pool, scalar_in])
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(64,  activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
out = tf.keras.layers.Dense(NUM_ACTIONS, activation='softmax', name='action')(x)

model = tf.keras.Model(
    inputs  = [my_hand_in, enemy_hand_in, scalar_in],
    outputs = out
)
model.compile(
    optimizer = tf.keras.optimizers.Adam(0.001),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

# パラメータ数の確認
total     = model.count_params()
trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
fixed     = total - trainable
print(f'総パラメータ数: {total:,}')
print(f'  学習するパラメータ: {trainable:,}  ← 誤差逆伝播で更新')
print(f'  固定パラメータ:     {fixed:,}  ← 数学知識（更新しない）')

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]
history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=60, callbacks=callbacks, verbose=1
)
print(f'\n最高検証精度: {max(history.history["val_accuracy"])*100:.1f}%')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title in zip(axes, ['accuracy','loss'], ['精度','損失']):
    ax.plot(history.history[key],         label='訓練')
    ax.plot(history.history[f'val_{key}'], label='検証')
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

# 学習Embeddingの可視化（AIがゲーム上の強さをどう学んだか）
weights  = model.get_layer('learned_emb').get_weights()[0]  # (151, 16)
strength = np.linalg.norm(weights, axis=1)
plt.figure(figsize=(14, 3))
plt.bar(range(1, 51), strength[1:51],
        color=['red' if all(n%p!=0 for p in [2,3,5,7]) else 'steelblue' for n in range(1,51)])
plt.title('AIが学習した数字の強さ（赤=素数系、青=合成数）1〜50')
plt.xlabel('数字'); plt.ylabel('学習ベクトルのノルム')
plt.show()

## TensorFlow.js 形式でエクスポート

In [ ]:
import os, shutil
OUT = 'tfjs_model'
if os.path.exists(OUT): shutil.rmtree(OUT)
tfjs.converters.save_keras_model(model, OUT)
print('エクスポート完了！')
for f in os.listdir(OUT):
    print(f'  {f} ({os.path.getsize(os.path.join(OUT,f))/1024:.1f} KB)')

shutil.make_archive('tfjs_model', 'zip', 'tfjs_model')
from google.colab import files
files.download('tfjs_model.zip')
print('ダウンロード開始！')

## 動作テスト

In [ ]:
names = ['パス','攻撃','足し算','引き算','商×余','桁和分裂','GCD掛け']
tests = [
    {'p1':[6,3,10,7,2],  'p2':[12,18,5,9,4],              'desc':'攻撃チャンスあり（6→12,18を奪える）'},
    {'p1':[5,8,3,2,7],   'p2':[1,1,1,1,1],                'desc':'合成が有利（相手が全部1）'},
    {'p1':[6,9,4,3,2],   'p2':[8,10,6,4,3],               'desc':'GCD有効（6と9のGCD=3→18）'},
    {'p1':[2,3],         'p2':[4,6,8,10,12,14,16,18],     'desc':'手札2枚 vs 相手8枚'},
    {'p1':[36,12,24,48], 'p2':[7,11,13,17,19,23],         'desc':'相手が全員素数'},
]

for t in tests:
    g = MathCardGame()
    g.reset()
    g.hands['P1'] = t['p1']; g.hands['P2'] = t['p2']
    g.turn_count = 3; g.scores = {'P1': 20, 'P2': 30}
    my_h, enm_h, sc = g.get_inputs('P1')
    pred = model.predict(
        {'my_hand': np.array([my_h]),
         'enemy_hand': np.array([enm_h]),
         'scalars': np.array([sc])},
        verbose=0
    )[0]
    chosen  = int(np.argmax(pred))
    optimal = g.get_optimal_action('P1')
    print(f'【{t["desc"]}】')
    print(f'  自分: {t["p1"]}  相手: {t["p2"]}')
    print(f'  AI: {names[chosen]} | 最適: {names[optimal]} | {"✅" if chosen==optimal else "❌"}')
    probs = ' '.join(f'{names[i]}:{pred[i]:.2f}' for i in range(NUM_ACTIONS))
    print(f'  {probs}'); print()